In [2]:
# 1. Create FrozenLake environment.
# 2. Collect offline data using a mixed behavior policy.
# 3. Convert states into one-hot vectors.
# 4. Build a Q-network with two hidden layers.
# 5. Train Offline DQN using Bellman loss.
# 6. Train CQL using Bellman loss + conservative penalty.
# 7. Evaluate both models using the highest Q-value action.
# 8. Compare Q-values for common and rare actions.
# 9. CQL reduces Q-value overestimation.

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

# -----------------------------
# 1. Setup
# -----------------------------
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)

N_STATES = 16
N_ACTIONS = 4

TERMINAL = {5, 7, 11, 12, 15}


# One-hot encoding
def one_hot(s):
    v = np.zeros(N_STATES, dtype=np.float32)
    v[s] = 1.0
    return v


# -----------------------------
# 2. Collect Offline Dataset
# -----------------------------
def behavior_policy():
    if np.random.rand() < 0.3:
        return env.action_space.sample()  # 30% random

    return np.random.choice([1, 2])       # 70% Down/Right


S, A, R, S2, D = [], [], [], [], []

for _ in range(300):

    s, _ = env.reset()
    done = False

    while not done:

        a = behavior_policy()

        s2, r, term, trunc, _ = env.step(a)

        done = term or trunc

        S.append(s)
        A.append(a)
        R.append(r)
        S2.append(s2)
        D.append(float(term))

        s = s2


S = np.array(S)
A = np.array(A)
S2 = np.array(S2)

R = np.array(R, dtype=np.float32)
D = np.array(D, dtype=np.float32)


# -----------------------------
# 3. Find Common and Rare Pairs
# -----------------------------
pair_counts = np.zeros(
    (N_STATES, N_ACTIONS),
    dtype=int
)

for s, a in zip(S, A):
    pair_counts[s, a] += 1


valid = [
    s for s in range(N_STATES)
    if s not in TERMINAL
]

common = [
    (s, a)
    for s in valid
    for a in range(4)
    if pair_counts[s, a] >= 5
]

rare = [
    (s, a)
    for s in valid
    for a in range(4)
    if pair_counts[s, a] < 5
]


# -----------------------------
# 4. Q-Network
# -----------------------------
class QNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(N_STATES, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, N_ACTIONS)
        )

    def forward(self, x):
        return self.net(x)


# -----------------------------
# 5. CQL Loss
# -----------------------------
def cql_loss(
    q_net,
    target_net,
    s,
    a,
    r,
    s2,
    done,
    gamma,
    alpha
):

    q_all = q_net(s)

    # Q-value of selected action
    q_data = q_all.gather(
        1,
        a.unsqueeze(1)
    ).squeeze(1)

    # Bellman target
    with torch.no_grad():

        target = (
            r
            + gamma * (1 - done)
            * target_net(s2).max(dim=1).values
        )

    # Bellman loss
    bellman = F.mse_loss(
        q_data,
        target
    )

    # Conservative Q-learning penalty
    cql_pen = (
        torch.logsumexp(q_all, dim=1)
        - q_data
    ).mean()

    return bellman + alpha * cql_pen


# -----------------------------
# 6. Convert Dataset to Tensors
# -----------------------------
S_T = torch.tensor(
    np.array([one_hot(x) for x in S])
)

S2_T = torch.tensor(
    np.array([one_hot(x) for x in S2])
)

A_T = torch.tensor(
    A,
    dtype=torch.long
)

R_T = torch.tensor(R)
D_T = torch.tensor(D)


# -----------------------------
# 7. Training Function
# -----------------------------
def train(
    alpha,
    steps=3000,
    batch=64,
    lr=1e-3,
    gamma=0.99
):

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    q = QNet()
    target = QNet()

    target.load_state_dict(
        q.state_dict()
    )

    optimizer = torch.optim.Adam(
        q.parameters(),
        lr=lr
    )

    for step in range(steps):

        indices = np.random.randint(
            0,
            len(S),
            batch
        )

        loss = cql_loss(
            q,
            target,
            S_T[indices],
            A_T[indices],
            R_T[indices],
            S2_T[indices],
            D_T[indices],
            gamma,
            alpha
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # Update target network
        if step % 100 == 0:
            target.load_state_dict(
                q.state_dict()
            )

    return q


# -----------------------------
# 8. Create Q-Table
# -----------------------------
def q_table(q):

    states = torch.tensor(
        np.array([
            one_hot(s)
            for s in range(N_STATES)
        ])
    )

    with torch.no_grad():
        return q(states).numpy()


# -----------------------------
# 9. Evaluate Model
# -----------------------------
def evaluate(q):

    s, _ = env.reset()

    total = 0.0
    done = False
    steps = 0

    while not done and steps < 50:

        with torch.no_grad():

            state = torch.tensor(
                one_hot(s)
            ).unsqueeze(0)

            a = q(state).argmax().item()

        s, r, term, trunc, _ = env.step(a)

        done = term or trunc

        total += r
        steps += 1

    return total


# -----------------------------
# 10. Average Q-value
# -----------------------------
def avg_q(qt, pairs):

    return float(
        np.mean([
            qt[s, a]
            for s, a in pairs
        ])
    )


# -----------------------------
# 11. Train Both Models
# -----------------------------
dqn_model = train(alpha=0.0)
cql_model = train(alpha=1.0)

dqn_q = q_table(dqn_model)
cql_q = q_table(cql_model)

dqn_ret = evaluate(dqn_model)
cql_ret = evaluate(cql_model)


# -----------------------------
# 12. Print Results
# -----------------------------
print(
    f"Dataset: {len(S)} transitions | "
    f"Left used {100*np.mean(A==0):.0f}%, "
    f"Down {100*np.mean(A==1):.0f}%, "
    f"Right {100*np.mean(A==2):.0f}%, "
    f"Up {100*np.mean(A==3):.0f}%"
)

print(
    f"Rare/unseen (state,action) pairs: "
    f"{len(rare)} of {len(rare)+len(common)}\n"
)

print(
    f"{'Model':<15}"
    f"{'Return':>8}"
    f"{'Avg Q common':>15}"
    f"{'Avg Q rare':>13}"
    f"{'Rare Q > 1.0':>15}"
)

for name, qt, ret in [
    ("Offline DQN", dqn_q, dqn_ret),
    ("CQL", cql_q, cql_ret)
]:

    over = sum(
        qt[s, a] > 1.0
        for s, a in rare
    )

    print(
        f"{name:<15}"
        f"{ret:>8.1f}"
        f"{avg_q(qt, common):>15.3f}"
        f"{avg_q(qt, rare):>13.3f}"
        f"{over:>11d}/{len(rare)}"
    )

print(
    "\nNote: Maximum possible reward is 1.0, "
    "so Q-values above 1.0 indicate overestimation."
)

Dataset: 1258 transitions | Left used 9%, Down 42%, Right 42%, Up 7%
Rare/unseen (state,action) pairs: 9 of 44

Model            Return   Avg Q common   Avg Q rare   Rare Q > 1.0
Offline DQN         1.0          0.771        0.748          0/9
CQL                 0.0          0.902       -0.180          0/9

Note: Maximum possible reward is 1.0, so Q-values above 1.0 indicate overestimation.
